# Quantum ESPRESSO SCF Calculation on HPC

Workflow:
HPC Environment → QE Input → SLURM Job → SCF Calculation → Energy Extraction

## 1. HPC Environment Check

In [ ]:
!hostname

import sys
sys.executable

In [ ]:
%%bash
module load materials/qe/7.2-openmpi
which pw.x

## 2. Create QE Working Directory

In [ ]:
from pathlib import Path
workdir = Path('qe_test')
workdir.mkdir(exist_ok=True)
print('Working directory:', workdir)

## 3. Generate Quantum ESPRESSO Input

In [ ]:
qe_input = """&CONTROL
 calculation='scf'
 prefix='si'
 pseudo_dir='/mgpfs/home/lala002/pseudo'
 outdir='/mgpfs/home/lala002/qe_test/tmp'
/
&SYSTEM
 ibrav=2
 celldm(1)=10.2
 nat=2
 ntyp=1
 ecutwfc=30
/
&ELECTRONS
 conv_thr=1.0d-8
/
ATOMIC_SPECIES
Si 28.0855 Si.pbe-n-kjpaw_psl.1.0.0.UPF
ATOMIC_POSITIONS crystal
Si 0.00 0.00 0.00
Si 0.25 0.25 0.25
K_POINTS automatic
4 4 4 0 0 0
"""

(workdir/'si.scf.in').write_text(qe_input)
print('QE input created')

## 4. Generate SLURM Script

In [ ]:
slurm = """#!/bin/bash
#SBATCH --job-name=QE-test
#SBATCH --partition=short
#SBATCH --nodes=1
#SBATCH --ntasks=16
#SBATCH --time=00:30:00
#SBATCH --output=qe-%j.out

module load materials/qe/7.2-openmpi
cd $SLURM_SUBMIT_DIR
mkdir -p tmp

mpirun --mca btl ^openib -np 16 pw.x -in si.scf.in > si.scf.out
"""

(workdir/'run_qe.sh').write_text(slurm)
print('SLURM script created')

## 5. Submit QE Calculation

In [ ]:
!cd qe_test && sbatch run_qe.sh

## 6. Monitor Job

In [ ]:
!squeue -u $USER

## 7. Extract Total Energy

In [ ]:
import re

text = open('qe_test/si.scf.out').read()
energy = re.findall(r'!\s+total energy\s+=\s+(-?\d+\.\d+)', text)

if energy:
    print("Total Energy =", energy[-1], "Ry")
else:
    print("Calculation result not found yet")